# A 股组合回测 Demo

本 Notebook 演示使用 `vectorbt_qs` 进行 A 股回测的完整流程：

1. 加载行情数据
2. 生成均线交叉策略的目标权重
3. 应用 A 股交易约束（停牌/涨跌停/费率）
4. 执行回测并分析结果

## 1. 环境准备

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# 确保项目根目录在 path 中
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))))

from mvp.data.adapter import load_ashare_daily_bar, load_ashare_calendar
from mvp.engine.runner import run_backtest, portfolio_report
import vectorbt as vbt

print("✅ 环境就绪")
print(f"   vectorbt 版本: {vbt.__version__}")

## 2. 参数配置

可调整回测参数：日期范围、初始资金、标的数量等。

In [ ]:
# ======== 可调参数 ========
MARKET = "ashare"
START = "2024-06-01"            # ← 1 年数据
END = "2025-06-01"
INIT_CASH = 1_000_000.0
MAX_SYMBOLS = 30

# 数据源: "cos" = COS直接读 | "local" = 本地 data/ 目录
DATA_SOURCE = "cos"

# 回测引擎参数
BACKTEST_CONFIG = {
    "init_cash": INIT_CASH,
    "size_granularity": 100,
    "slippage": 0.001,
    "freq": "1D",
}

# 切换数据源
if DATA_SOURCE == "cos":
    from mvp.data.adapter import set_data_root
    set_data_root("ashare", "cos://qs-cold/clean_data/ashare/lqtp_data")
    print("⚠️  首次从 COS 读取会下载并缓存 (~2-5 分钟/年)")
else:
    print("📂 使用本地 data/ 目录")

print(f"📅 回测区间: {START} ~ {END}")
print(f"💰 初始资金: ¥{INIT_CASH:,.0f}")
print(f"📊 最多标的: {MAX_SYMBOLS}")

## 3. 加载行情数据

从 COS parquet 加载 `StockDailyBar`，自动拼接为 pandas 宽表。输出包含复权 OHLCV、停牌标记、涨跌停价等。

In [ ]:
# 加载数据
data = load_ashare_daily_bar(symbols=None, start=START, end=END)

# 使用复权收盘价
close = data["Close_adj"]
is_suspend = data["is_suspend"]
high_limit = data["high_limit"]
low_limit = data["low_limit"]

# 限制标的数量
if MAX_SYMBOLS and close.shape[1] > MAX_SYMBOLS:
    top_symbols = close.iloc[-1].dropna().sort_values(ascending=False).head(MAX_SYMBOLS).index
    close = close[top_symbols]
    is_suspend = is_suspend[top_symbols] if not is_suspend.empty else pd.DataFrame(False, index=close.index, columns=close.columns)
    high_limit = high_limit[top_symbols] if not high_limit.empty else close * 1.10
    low_limit = low_limit[top_symbols] if not low_limit.empty else close * 0.90

print(f"📊 数据形状: {close.shape[0]} 天 × {close.shape[1]} 只")
print(f"📅 日期范围: {close.index[0].date()} ~ {close.index[-1].date()}")
print(f"\n前 5 只标的:\n{list(close.columns[:5])}")
print(f"\n复权收盘价预览:")
close.head()

## 4. 生成目标权重

3 种模拟方式可选：

| 模式 | 说明 |
|------|------|
| `random` | 每天随机选 10 只，各 5% |
| `index_like` | 类似指数持仓 + 噪声 + 低换手 |
| `ma_cross` | 均线交叉信号 |

> 💡 替换这里即可接入你自己的因子模型输出。

In [ ]:
# ======== 选择权重生成模式 ========
WEIGHT_MODE = "random"  # 'random' | 'index_like' | 'ma_cross'

def random_weights(close, n_stocks=10, weight=0.05, seed=42):
    """每天随机选 n_stocks 只做多"""
    rng = np.random.default_rng(seed)
    weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)
    for i, date in enumerate(close.index):
        chosen = rng.choice(close.columns, size=min(n_stocks, len(close.columns)), replace=False)
        weights.loc[date, chosen] = weight
    return weights

def index_like_weights(close, n_stocks=10, noise_std=0.01, turnover=0.15, weight=0.05, seed=42):
    """指数型持仓：稳定池 + 噪声 + 低换手"""
    rng = np.random.default_rng(seed)
    weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)
    pool = rng.choice(close.columns, size=n_stocks, replace=False).tolist()
    
    for i, date in enumerate(close.index):
        if rng.random() < turnover or i == 0:
            pool = rng.choice(close.columns, size=n_stocks, replace=False).tolist()
        for ticker in pool:
            noise = rng.normal(0, noise_std)
            weights.loc[date, ticker] = max(0, weight + noise)
    return weights

def ma_cross_weights(close, fast=10, slow=30, weight=0.05):
    """均线交叉"""
    fast_ma = close.rolling(fast).mean()
    slow_ma = close.rolling(slow).mean()
    weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)
    weights[fast_ma > slow_ma] = weight
    return weights.iloc[slow:]

# 生成
if WEIGHT_MODE == "random":
    target_weights = random_weights(close)
elif WEIGHT_MODE == "index_like":
    target_weights = index_like_weights(close)
else:
    target_weights = ma_cross_weights(close)
    target_weights = target_weights.reindex(close.index).fillna(0.0)

avg_pos = (target_weights > 0).sum(axis=1).mean()
print(f"📈 目标权重: {target_weights.shape}, 日均持仓 {avg_pos:.1f} 只")

# 展示某一天
sample_date = target_weights.index[len(target_weights)//2]
s = target_weights.loc[sample_date]
print(f"\n📅 {sample_date.date()} 截面 (前 10):")
s[s > 0].head(10)

## 5. 执行回测

调用 `run_backtest`，自动串联：数据对齐 → A 股约束（停牌/涨跌停/费率） → vectorbt 引擎。

In [ ]:
# 执行回测
pf = run_backtest(
    MARKET,
    target_weights,
    config=BACKTEST_CONFIG,
)

print("✅ 回测完成\n")
print(f"   总交易笔数: {len(pf.trades.records_readable)}")
print(f"   最终净值:   ¥{pf.final_value():,.0f}")

## 6. 绩效报告

In [ ]:
# 核心绩效指标
report = portfolio_report(pf)
for metric, value in report.items():
    print(f"  {metric:30s}: {value}")
    
# 完整的 stats（几十个指标）
print("\n── 完整 stats ──")
pf.stats()

## 7. 可视化

交互式图表（Plotly），支持缩放和悬停详情。

In [ ]:
# 净值曲线
pf.plot_value(title="A 股组合净值 — 均线交叉策略").show()

In [ ]:
# 回撤曲线
pf.plot_drawdowns(title="回撤").show()

In [ ]:
# 仓位热力图（每只股票每天的持仓占比）
pf.plot_asset_value(title="持仓市值分布").show()

## 8. 交易明细

In [ ]:
# 每笔交易的完整记录
trades = pf.trades.records_readable

if len(trades) > 0:
    print(f"📊 共 {len(trades)} 笔交易\n")
    # 按 PnL 排序，看最佳和最差
    display_cols = ["Entry Timestamp", "Exit Timestamp", "Size", "PnL", "Return", "Direction"]
    print("🏆 最佳 5 笔:")
    display(trades.nlargest(5, "PnL")[display_cols])
    print("\n💸 最差 5 笔:")
    display(trades.nsmallest(5, "PnL")[display_cols])
else:
    print("⚠️ 无成交记录")